In [ ]:
[dim_finemap]
parameter: pheno_files = paths         
parameter: ld_dir = path               
parameter: B_stoc = 10000               
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "72h"
parameter: mem = "60G"
parameter: numThreads = 2
parameter: container = ""
input: pheno_files, group_by = 1
output: f'{cwd:a}/finemap/{_input:bn}_B_stoc_{B_stoc}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    ## ---- safe wrapper (yours) ----------------------------------------------
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(.fn(...), warning = function(w) invokeRestart("muffleWarning")),
        error = function(e) { cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e))); NULL }
      )
    }

    ## ---- load matched pheno + LD -------------------------------------------
    pheno_path <- ${_input:r}
    pheno_dat  <- readRDS(pheno_path)

    region_tok <- str_extract(basename(pheno_path), "region_[0-9]+")
    ld_path    <- file.path("${ld_dir}", paste0(region_tok, "_ukb_B_stoc_", ${B_stoc}, ".rds"))
    stopifnot(file.exists(ld_path))
    ld_dat     <- readRDS(ld_path)

    z        <- pheno_dat$z_vec               # == sumstat_qc$z
    n        <- pheno_dat$n                    # full GWAS sample size (342053)
    true_vars<- pheno_dat$true_vars

    LD_full  <- ld_dat$LD_full
    LD_sub   <- ld_dat$LD_sample
    U        <- ld_dat$U                       # p x B_stoc
    B_sketch <- ncol(U)                        # = ld_dat$B_stoc, sketch dimension
    n_sub    <- length(ld_dat$idx_ld)          # subset sample size behind LD_sample

    ## ---- 5 finemapping runs -------------------------------------------------
    # 1. in-sample LD (exact)
    fit_insample   <- safe_susie(susie_hc, "insample",   z = z, R = LD_full, n = n)

    # 2. subset LD (naive, no correction)
    fit_subset     <- safe_susie(susie_hc, "subset",     z = z, R = LD_sub,  n = n)

    # 3. sketch (naive, U as input, no correction)
    fit_sketch     <- safe_susie(susie_hc, "sketch",     z = z, X = t(U),    n = n)

    # 4. sketch + correction   -> R_finite = sketch dimension
    fit_sketch_mix <- safe_susie(susie_hc, "sketch_mix", z = z, X = t(U),    n = n,
                                 R_finite = B_sketch)

    # 5. subset LD + correction -> R_finite = subset sample size
    fit_subset_mix <- safe_susie(susie_hc, "subset_mix", z = z, R = LD_sub,  n = n,
                                 R_finite = n_sub)

    ## ---- save ---------------------------------------------------------------
    finemap_out <- list(
      region_id      = pheno_dat$region_id,
      h2g            = pheno_dat$h2g,
      n_causal       = pheno_dat$n_causal,
      B_stoc         = B_sketch,
      n_sub          = n_sub,
      true_vars      = true_vars,
      fit_insample   = fit_insample,
      fit_subset     = fit_subset,
      fit_sketch     = fit_sketch,
      fit_sketch_mix = fit_sketch_mix,
      fit_subset_mix = fit_subset_mix
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(finemap_out, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B_finemap"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

work_dir="${job_dir}"

# ===== BSUB template for the finemap step =====
cat << 'EOF' > ${job_name}_finemap.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/05.Fig2B.Sketch_Dimension_Finemap.ipynb" dim_finemap \
    --pheno_files $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/h2g_H2G/causal_CAUSAL/pheno/*.rds) \
    --ld_dir /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/B_B_STOC/ld \
    --B_stoc B_STOC \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/B_B_STOC/ -j 5
EOF

chmod +x ${job_name}_finemap.sh

for causal in 1 2 3; do
  for h2g in 0.0004 0.0006 0.0008; do
    for B_stoc in 500 1000 2000 3000 5000 10000 20000; do
        output_script="${job_name}_finemap_causal_${causal}_h2g_${h2g}_B_stoc_${B_stoc}.sh"
        sed -e "s|CAUSAL|${causal}|g" \
            -e "s|H2G|${h2g}|g" \
            -e "s|B_STOC|${B_stoc}|g" \
            -e "s|WORK_DIR|${work_dir}|g" \
            -e "s|JOB_NAME|${job_name}|g" \
          "${job_name}_finemap.sh" > "${output_script}"

         sb "${output_script}"
    done
  done
done

In [ ]:
[susie_cs_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_summary.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # ---- CS-level helpers --------------------------------------------------
    true_set_num <- function(variant, cs) {                 # # CS hitting a causal (TP at CS level)
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(x) length(intersect(variant, x)) > 0, logical(1)))
    }
    discover_var_num <- function(variant, cs) {             # # causals captured in the CS union
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }

    # ---- aggregate all finemap outputs in `folder` -------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "\\.rds$", recursive = TRUE, full.names = TRUE)

    tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars

      # metadata from stored fields, fall back to filename
      ncausal <- res$n_causal %||% as.numeric(sub(".*ncausal_([0-9.]+).*", "\\1", basename(f)))
      h2g     <- res$h2g      %||% as.numeric(sub(".*h2g_([0-9.eE+-]+?)(_B_stoc.*)?\\.rds$", "\\1", basename(f)))
      B_stoc  <- res$B_stoc   %||% as.numeric(sub(".*B_stoc_([0-9]+).*", "\\1", basename(f)))

      # methods = the fit_* fields (fall back to a result_list if present)
      if (!is.null(res$result_list)) {
        result_list <- res$result_list
      } else {
        fit_names   <- grep("^fit_", names(res), value = TRUE)
        result_list <- setNames(res[fit_names], sub("^fit_", "", fit_names))
      }

      row_from_fit <- function(fit, label) {
        cs <- fit$sets$cs
        tibble(
          input_file       = f,
          method           = label,
          ncausal          = ncausal,
          h2g              = h2g,
          B_stoc           = B_stoc,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs)),
          B_corrected      = fit$B_corrected %||% NA_real_
        )
      }

      map_dfr(names(result_list), function(m) {
        sus  <- result_list[[m]]
        if (is.null(sus)) return(NULL)               # safe_susie NULLs
        rows <- row_from_fit(sus, m)
        ser  <- sus$R_finite_diagnostics$ser_model    # SER fallback sub-model, if any
        if (!is.null(ser)) rows <- bind_rows(rows, row_from_fit(ser, paste0(m, "_SER")))
        rows
      })
    })
    saveRDS(tb, ${_output:r})

In [ ]:
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/05.Fig2B.Sketch_Dimension_Finemap.ipynb" susie_cs_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/result \
    --cwd    /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/summary

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J B_dim_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/05.Fig2B.Sketch_Dimension_Finemap.ipynb" susie_cs_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
res = readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/summary/result_susie_cs_summary_summary.rds")

library(tidyverse)
summ = res %>%
  group_by(method, h2g, B_stoc, total_var_number) %>%
  summarize(
    true_var_number     = sum(true_var_number),
    true_set_number     = sum(true_set_number),
    total_causal_number = sum(total_var_number),
    total_set_number    = sum(total_set_number),
    rep                 = n(),
    total_set_length    = sum(total_set_length),
    .groups = "drop"
  ) %>%
  mutate(
    power = true_var_number / total_causal_number,
    FDR   = if_else(total_set_number == 0, 0,
                    1 - (true_set_number / total_set_number)),
    ave_size = total_set_length / total_set_number,
    f1_score = if_else(
      power == 0 & FDR == 1,
      0,
      power * (1 - FDR) / (power + 1 - FDR)
    )
  )

library(tidyverse)

fig_data_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_QC/Figure_data"
fig_out_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_QC/Figure_output"

## proposed eb_mix methods LAST (rightmost); colors keyed by name
method_lv <- c("insample","subset","sketch","subset_mix","sketch_mix")
rss_pal   <- c(insample="#164B70", subset="#56B4E9", sketch="#0072B2",
               subset_mix="#CA705E", sketch_mix="#B24745")

## ---- plot-ready data ---------------------------------------------------
plot_dt <- summ %>%
  mutate(
    B_stoc = as.integer(str_extract(as.character(B_stoc), "[0-9]+")),
    method = factor(method, levels = method_lv),
    h2g    = factor(h2g),
    B_lab  = factor(paste0("B = ", B_stoc), levels = paste0("B = ", sort(unique(B_stoc)))),
    varnum = factor(total_var_number)
  )
saveRDS(plot_dt, file.path(fig_data_dir, "cs_metrics_by_B_plotdt.rds"))
plot_dt <- readRDS(file.path(fig_data_dir, "cs_metrics_by_B_plotdt.rds"))

theme_clean <- theme_bw(base_size = 16) +
  theme(panel.grid = element_blank(), strip.background = element_blank(),
        strip.text = element_text(face = "bold"), axis.line = element_line(color = "black"),
        panel.spacing = unit(1.2, "lines"), legend.position = "right")

base_bar <- function(df, yvar, ylab) {
  ggplot(df, aes(varnum, .data[[yvar]], fill = method)) +
    geom_col(position = position_dodge(width = 0.8), width = 0.7) +
    facet_grid(h2g ~ B_lab) +
    scale_fill_manual(values = rss_pal, breaks = method_lv) +
    labs(x = "number of causal variants", y = ylab, fill = "method") +
    theme_clean
}

## ---- POWER -------------------------------------------------------------
X <- 25; Y <- 6
options(repr.plot.width = X, repr.plot.height = Y)
fig_name <- "power_bar_facet_h2g_B"
p_power  <- base_bar(plot_dt, "power", "Power") + coord_cartesian(ylim = c(0, 1))
p_power
# ggsave(file.path(fig_out_dir, paste0(fig_name, ".png")), p_power, width = X, height = Y, dpi = 300)
# ggsave(file.path(fig_out_dir, paste0(fig_name, ".pdf")), p_power, width = X, height = Y, device = cairo_pdf)

## ---- FDR ---------------------------------------------------------------
X <- 25; Y <- 6
options(repr.plot.width = X, repr.plot.height = Y)
fig_name <- "fdr_bar_facet_h2g_B"
p_fdr    <- base_bar(plot_dt, "FDR", "FDR") +
  geom_hline(yintercept = 0.05, linetype = "dashed", color = "grey50") +
  coord_cartesian(ylim = c(0, 1))
p_fdr
# ggsave(file.path(fig_out_dir, paste0(fig_name, ".png")), p_fdr, width = X, height = Y, dpi = 300)
# ggsave(file.path(fig_out_dir, paste0(fig_name, ".pdf")), p_fdr, width = X, height = Y, device = cairo_pdf)